# 01 — Understanding raw M5 data

Goal of this notebook: understand *why* the ingestion pipeline (`src/pipeline/ingest.py`)
does what it does, by doing it manually, in small steps, on real data, with printed
output at every step.

We are NOT trying to reproduce the full pipeline here. We're looking at small samples
so you can actually read the output and reason about it.

Update `RAW_DIR` below to point at your `data/raw/` folder.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")  # adjust if your notebook lives elsewhere
pd.set_option("display.max_columns", 15)


## Step 1 — Look at sales_train_validation.csv raw, unmodified

This is the file with the confusing `d_1`, `d_2`, ... `d_1913` columns.
Load just a handful of rows and columns — don't try to load the whole thing yet.

In [2]:
sales_raw = pd.read_csv(RAW_DIR / "sales_train_validation.csv", nrows=5)
sales_raw.iloc[:, :10]  # first 10 columns only, this file has ~1919 columns total


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0


**What are you looking at?**

- `id`, `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` — these identify *which
  series* this row belongs to. One row = one item at one store, for its ENTIRE history.
- `d_1`, `d_2`, `d_3`, ... — these are **not dates**. They are day *counters*. `d_1` is
  the first day in the whole M5 dataset's history, `d_2` is the next day, and so on,
  all the way to `d_1913`.

**Question to answer yourself before moving on:** if `d_1` is not a real date, how would
you ever know what day of the week, month, or year `d_500` actually happened on? Where
would that information have to come from?

(Answer is in Step 2 — try to guess first.)

In [3]:
print("Total columns:", sales_raw.shape[1])
print("How many are d_ day columns?", sum(c.startswith('d_') for c in sales_raw.columns))
print("First day column:", [c for c in sales_raw.columns if c.startswith('d_')][0])
print("Last day column:", [c for c in sales_raw.columns if c.startswith('d_')][-1])


Total columns: 1919
How many are d_ day columns? 1913
First day column: d_1
Last day column: d_1913


## Step 2 — calendar.csv is the decoder ring

This file maps every `d_` number to a real date, plus extra info for that date
(day of week, month, whether there was an event, SNAP welfare-benefit days).

This is the missing piece from Step 1.

In [4]:
calendar = pd.read_csv(RAW_DIR / "calendar.csv")
calendar.head(10)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1
5,2011-02-03,11101,Thursday,6,2,2011,d_6,NaN,NaN,NaN,NaN,1,1,1
6,2011-02-04,11101,Friday,7,2,2011,d_7,NaN,NaN,NaN,NaN,1,0,0
7,2011-02-05,11102,Saturday,1,2,2011,d_8,NaN,NaN,NaN,NaN,1,1,1
8,2011-02-06,11102,Sunday,2,2,2011,d_9,SuperBowl,Sporting,NaN,NaN,1,1,1
9,2011-02-07,11102,Monday,3,2,2011,d_10,NaN,NaN,NaN,NaN,1,1,0


In [5]:
# confirm the d -> date mapping directly
print(calendar[calendar['d'] == 'd_1'][['d', 'date', 'weekday']])
print(calendar[calendar['d'] == 'd_500'][['d', 'date', 'weekday']])
print(calendar[calendar['d'] == 'd_1913'][['d', 'date', 'weekday']])


     d        date   weekday
0  d_1  2011-01-29  Saturday
         d        date weekday
499  d_500  2012-06-11  Monday
           d        date weekday
1912  d_1913  2016-04-24  Sunday


Now you can answer the Step 1 question: `d_500` is meaningless on its own. You
MUST join against `calendar` to know it corresponds to a specific real date and
weekday. This is exactly why `ingest.py` does:

```sql
JOIN calendar c ON s.d = c.d
```

Without this join, you'd have sales numbers with no time context at all — no way to
build "is this a weekend", "is this near a holiday", or any date-based feature.

## Step 3 — why unpivot (wide -> long)?

Right now, one row = one item-store's ENTIRE history (1913 sales values crammed into
1913 columns). This is compact for storage, but useless for modeling:

- You can't easily add a `date` column per observation
- You can't easily compute "sales 7 days ago" as a feature
- Standard ML libraries expect one row per (thing you're predicting), not one row per
  entity with 1913 target values sitiing in separate columns

We need: one row per (item, store, single day). Let's do this manually on a tiny
slice so you can see the transformation directly.

In [6]:
# take just 3 items and their first 10 days, to keep this readable
sample = sales_raw[['id', 'item_id', 'store_id']].copy()
day_cols = [f"d_{i}" for i in range(1, 11)]
sample[day_cols] = pd.read_csv(RAW_DIR / "sales_train_validation.csv", nrows=5)[day_cols]
sample


,id,item_id,store_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,CA_1,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,CA_1,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,CA_1,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,CA_1,0,0,0,0,0,0,0,0,0,0


In [7]:
# this is the unpivot / "melt" — wide columns become row values
long_sample = sample.melt(
    id_vars=['id', 'item_id', 'store_id'],
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)
long_sample.sort_values(['id', 'd']).head(15)


,id,item_id,store_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_1,0
45,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_10,0
5,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_2,0
10,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_3,0
15,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_4,0
20,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_5,0
25,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_6,0
30,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_7,0
35,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_8,0
40,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,d_9,0


Compare shapes: the wide sample had 5 rows x 10 day-columns. The long version has
5 x 10 = 50 rows, one per (item-store, day). This is the shape ML models actually want.

`ingest.py` does the same operation in SQL (`UNPIVOT`) instead of pandas `melt`,
because doing this on the full 30,490 series x 1,913 days (~58 million rows) in
pandas would be slow and memory-heavy. DuckDB's UNPIVOT does the same logical
operation but is built for exactly this scale.

## Step 4 — why prices need a SEPARATE join (not just another column)

Sales are recorded **daily**. Prices are recorded **weekly** (`sell_prices.csv` has
one price per item-store-week, not per item-store-day). This is a different grain,
so it can't just be another column added during the unpivot — it needs its own join,
on a different key (`wm_yr_wk`, the week identifier).

In [8]:
prices = pd.read_csv(RAW_DIR / "sell_prices.csv", nrows=10)
prices


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
5,CA_1,HOBBIES_1_001,11330,8.26
6,CA_1,HOBBIES_1_001,11331,8.26
7,CA_1,HOBBIES_1_001,11332,8.26
8,CA_1,HOBBIES_1_001,11333,8.26
9,CA_1,HOBBIES_1_001,11334,8.26


In [9]:
# calendar has the wm_yr_wk column too - this is the shared key
calendar[['date', 'd', 'wm_yr_wk']].head(10)


,date,d,wm_yr_wk
0,2011-01-29,d_1,11101
1,2011-01-30,d_2,11101
2,2011-01-31,d_3,11101
3,2011-02-01,d_4,11101
4,2011-02-02,d_5,11101
5,2011-02-03,d_6,11101
6,2011-02-04,d_7,11101
7,2011-02-05,d_8,11102
8,2011-02-06,d_9,11102
9,2011-02-07,d_10,11102


So the full join chain is:

```
sales (daily, by d)  --join on d-->  calendar (has date + wm_yr_wk)  --join on item+store+wm_yr_wk-->  prices (weekly)
```

This is exactly `build_fact_table()` in `ingest.py`. Two joins, two different keys,
because sales and prices live at two different time grains.

**Why LEFT JOIN for prices specifically?** Some items didn't exist yet / weren't sold
in early weeks of the dataset, so they have no price for those weeks. A LEFT JOIN keeps
those sales rows and just gives them a NULL price, instead of silently dropping them
(which an INNER JOIN would do).

## Check your understanding

Before moving to notebook 02 (feature engineering), you should be able to answer:

1. What does `d_500` mean on its own, without any other file? What does it mean after
   joining `calendar`?
2. Why is one row = one (item, store, day) the right shape for modeling, instead of
   one row = one (item, store) with 1913 columns?
3. Why are prices joined on `wm_yr_wk` instead of `d`?
4. Why LEFT JOIN prices instead of INNER JOIN?

If any of these are still fuzzy, re-run the cells above and print more examples —
that's more useful right now than moving forward.